# Arabic trace rankings on **deduplicated** train/val, all methods, saved in order (CheckThat! 2026 Task 2)

Same as `saving-ranked-arabic-traces`, but train/val are **deduplicated first** with the rules of
`arabic-mlp-dedup.ipynb`. The ranker is then refit on the deduplicated train/val, so no test ranking uses a
train/val copy of a test claim. Every method of the Recall@5 table is run and the full trace order of each row
is saved to JSON.

**Deduplication (exact claim string; the official test set is never changed)**

| Step | Split | Rule |
|:-:|:--|:--|
| 1 | test | **nothing removed** (repeats inside test are kept) |
| 2 | train, val | remove rows whose claim occurs in test |
| 3 | train | keep the first occurrence (lowest row) of each claim |
| 4 | val | remove rows whose claim occurs in the deduplicated train |
| 5 | val | keep the first occurrence (lowest row) of each claim |

Afterwards the notebook asserts that train, val and test share no claim and that train and val contain no
repeated claim. Removed rows are listed in `dedup_removed_items.csv`.

| Method (file name) | How traces are ordered |
|:--|:--|
| `random` | random permutation (seed 0) |
| `majority_first` | traces of the most frequent verdict first |
| `minority_first` | traces of the least frequent verdict first |
| `claim_cosine` | cosine(claim, trace) with the Qwen3-Embedding-8B vectors |
| `pn_counts` | p/n rule, label probabilities from verdict counts (+ exact claim memory) |
| `pn_selected` | p/n rule, feature blocks chosen by 5-fold CV (**ours**) |
| `pn_selected_shift` | `pn_selected` + covariate-shift weights (‡ chosen after seeing test) |
| `oracle` | gold label known: relevant traces first (upper bound) |

**Splits.** *Test:* models are fit on train+val and applied to all 511 test rows.
*Train / val:* out-of-fold rankings. Train+val is split into 5 folds and every row is ranked by a model
fitted on the other 4 folds, so no learned ranking uses its own label (the oracle uses the gold label by design).

**Row ids.** Train/val rows keep their **original position** in the released files (`row` in the output
records, and the `id_<row>` key of the embedding files), so they can be matched to the original data.
Test rows keep their position and `query_id`.

**Labels.** The Arabic gold labels are only *True* / *False* (no gold *Conflicting*), so the label model
has two classes. *Conflicting* still counts as a trace verdict: such traces are never relevant and rank last
under the p/n rule.

**Outputs** (in `OUT_DIR`):
* `rankings/<method>/{train,val,test}.json`: one record per row, traces in rank order with text, verdict and relevance
* `recall_at_k.csv`: Recall@1..5 (all rows) per method and split
* `run_meta.json`: input paths, row counts, CV scores and the selected feature blocks
* `dedup_report.json`, `dedup_removed_items.csv`: deduplication steps, kept rows, removed rows with reason

**Run on Kaggle:** accelerator **None (CPU)** is enough, internet not needed. Attach the CheckThat! data and
the Arabic embeddings (`{train,val,test}_qwen3emb8b_ar_v1.npz`), check the two paths in the next cell, then *Run All*.
About 5–10 min.

## 1. Configuration

In [ ]:
# ---------------- CHANGE THESE TWO PATHS IF NEEDED ----------------
DATA_ROOT = "/kaggle/input/datasets/d202511054/clef-2026-checkthat-task-2-data"  # contains Arabic/clef2026_gpt4_o_mini_{train,val}_arabic.json, ...
EMB_DIR   = "/kaggle/input/datasets/d202511054/clef-2026-checkthat-task-2-data/qwen3 arabic embeddings"  # contains {train,val,test}_qwen3emb8b_ar_v1.npz
# -------------------------------------------------------------------
OUT_DIR = "/kaggle/working/ar_dedup_trace_rankings"

SAVE_TRACE_TEXT    = True    # False = save the order without the trace text (much smaller files)
REQUIRE_EMBEDDINGS = True    # False = if the npz files are missing, run counts + memory only (no cosine / CV blocks)
PCA_DIM    = 32              # PCA size of claim / group-centroid embeddings
MIN_GAIN   = 0.001           # CV R@5 margin an extra feature block must win by
SHIFT_CLIP = 5.0             # covariate-shift weight clip
N_FOLDS    = 5
K          = 5               # Recall@K
SEED       = 0

## 2. Setup and input files

In [ ]:
import glob, json, os, re, time, warnings
from collections import Counter

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
CLASSES = ["true", "false", "conflicting"]
SPLITS  = ("train", "val", "test")
T0 = time.time()

JSON_NAMES = ("clef2026_gpt4_o_mini_train_arabic.json", "clef2026_gpt4_o_mini_val_arabic.json",
              "clef_arabic_test_gold_labels-final.json")
NPZ_NAME   = "{split}_qwen3emb8b_ar_v1.npz"

def find_one(root, name):
    """Recursive search for a file name; Arabic folders are preferred, English/Spanish ones are ignored
    (the Spanish embeddings carry the same '_ar_v1' file names)."""
    hits = [h for h in glob.glob(os.path.join(root, "**", name), recursive=True)
            if "span" not in h.lower() and "english" not in h.lower()]
    hits.sort(key=lambda h: ("arab" not in h.lower(), len(h)))
    return hits[0] if hits else None

if not os.path.isdir(DATA_ROOT):
    raise FileNotFoundError(f"DATA_ROOT {DATA_ROOT} does not exist - attach the dataset and fix the path in §1")
JSON_PATHS = {s: find_one(DATA_ROOT, f) for s, f in zip(SPLITS, JSON_NAMES)}
NPZ_PATHS  = {s: find_one(EMB_DIR, NPZ_NAME.format(split=s)) if os.path.isdir(EMB_DIR) else None for s in SPLITS}
for s in SPLITS:
    print(f"{s:5s} json: {JSON_PATHS[s]}\n      npz : {NPZ_PATHS[s]}")
print("\nall *_qwen3emb8b_ar_v1.npz files visible under DATA_ROOT / EMB_DIR (the ones above are used;"
      " Spanish folders hold files with the same names and are skipped):")
for h in sorted({h for r in (DATA_ROOT, EMB_DIR) if os.path.isdir(r)
                 for h in glob.glob(os.path.join(r, "**", NPZ_NAME.format(split="*")), recursive=True)}):
    print(f"  {os.path.getsize(h):>14,d} bytes  {h}")
if not all(JSON_PATHS.values()):
    raise FileNotFoundError(f"Arabic JSON files not found under {DATA_ROOT}: {JSON_PATHS}")
USE_EMB = all(NPZ_PATHS.values())
if not USE_EMB:
    if REQUIRE_EMBEDDINGS:
        raise FileNotFoundError(f"Embeddings not found under {EMB_DIR}: {NPZ_PATHS}. Fix EMB_DIR, "
                                f"or set REQUIRE_EMBEDDINGS=False to run counts + memory only.")
    print("WARNING: no embeddings - claim_cosine and the embedding feature blocks are skipped")
os.makedirs(OUT_DIR, exist_ok=True)

## 3. Load the Arabic data and deduplicate train/val
The five steps of the table above, in order, matching claims by exact string. Test is used exactly as released.

In [ ]:
def load_json(path):
    with open(path, encoding="utf-8") as f:
        s = f.read()
    try:
        return json.loads(s)
    except json.JSONDecodeError:
        # Same repair as the English notebook, in case a copy has a missing '"Reasoning_traces": [' key
        print(f"  {os.path.basename(path)}: invalid JSON, repairing the missing 'Reasoning_traces' key")
        return json.loads(re.sub(r'\{\s*\nx?(\s+)"(?![A-Za-z_]+"\s*:)', r'{\n        "Reasoning_traces": [\n\1"', s))

def norm(v):
    return str(v).strip().lower()

def claim_key(text):
    return re.sub(r"\s+", " ", str(text)).strip().lower()

class Data:
    """Arrays for one split. Embedding-derived fields stay None when no npz is available."""

    def __init__(self, items):
        self.items = items
        self.verdicts = [[norm(v) for v in e["Verdict_list"]] for e in items]
        self.gold = [norm(e.get("label", e.get("Label", ""))) for e in items]
        self.y = np.array([CLASSES.index(g) if g in CLASSES else -1 for g in self.gold])
        self.N = np.array([[sum(v == c for v in V) for c in CLASSES] for V in self.verdicts])
        self.qid = [e.get("query_id", e.get("query_index", i)) for i, e in enumerate(items)]
        self.keys = np.array([claim_key(e["claim"]) for e in items], dtype=object)
        self.orig_row = list(range(len(items)))   # position in the released file (set after deduplication)
        self.claim = None       # (n, dim) L2-normalised claim embeddings
        self.claim_pca = None   # (n, d)
        self.group = None       # (n, 9 + 3d) per-verdict-group features
        self.rep = None         # per claim: per-trace representativeness (within-group tie-break)
        self.claim_cos = None   # per claim: per-trace claim cosine

    def take(self, idx):
        d = Data.__new__(Data)
        for k, v in self.__dict__.items():
            if isinstance(v, np.ndarray):
                setattr(d, k, v[idx])
            elif isinstance(v, list):
                setattr(d, k, [v[i] for i in idx])
            else:
                setattr(d, k, v)
        return d

def concat(a, b):
    d = Data.__new__(Data)
    for k, v in a.__dict__.items():
        w = getattr(b, k)
        if isinstance(v, np.ndarray) and isinstance(w, np.ndarray):
            setattr(d, k, np.concatenate([v, w]))
        elif isinstance(v, list) and isinstance(w, list):
            setattr(d, k, v + w)
        else:
            setattr(d, k, None)
    return d

RAW  = {s: load_json(JSON_PATHS[s]) for s in SPLITS}
KEEP = {s: list(range(len(RAW[s]))) for s in SPLITS}          # kept original row positions
REMOVED, STEP_LOG = [], []

def _claims(s):
    return {RAW[s][i]["claim"] for i in KEEP[s]}

def _log(step, desc):
    STEP_LOG.append({"step": step, "description": desc, **{s: len(KEEP[s]) for s in SPLITS}})

def _remove(s, i, step, reason):
    e = RAW[s][i]
    REMOVED.append({"split": s, "row": i, "step": step, "reason": reason,
                    "label": e.get("label", e.get("Label")), "claim": e["claim"]})

def drop_if(s, step, reason, pred):
    keep = []
    for i in KEEP[s]:
        if pred(RAW[s][i]["claim"]): _remove(s, i, step, reason)
        else: keep.append(i)
    KEEP[s] = keep

def keep_first(s, step, reason):
    seen, keep = set(), []
    for i in KEEP[s]:                          # ascending row -> the lowest row is kept
        c = RAW[s][i]["claim"]
        if c in seen: _remove(s, i, step, reason)
        else: seen.add(c); keep.append(i)
    KEEP[s] = keep

_log(0, "original")
_log(1, "test: unchanged (official test set kept as released)")
TEST_CLAIMS = _claims("test")
drop_if("train", 2, "claim occurs in test", lambda c: c in TEST_CLAIMS)
drop_if("val",   2, "claim occurs in test", lambda c: c in TEST_CLAIMS)
_log(2, "train/val: drop claims seen in test")
keep_first("train", 3, "repeated claim within train")
_log(3, "train: keep first occurrence")
TRAIN_CLAIMS = _claims("train")
drop_if("val", 4, "claim occurs in deduplicated train", lambda c: c in TRAIN_CLAIMS)
_log(4, "val: drop claims seen in train")
keep_first("val", 5, "repeated claim within val")
_log(5, "val: keep first occurrence")

# post-conditions (fail loudly instead of ranking leaky data)
_C = {s: _claims(s) for s in SPLITS}
assert KEEP["test"] == list(range(len(RAW["test"]))), "test must not change"
assert not (_C["train"] & _C["test"]) and not (_C["val"] & _C["test"]), "test still shares claims with train/val"
assert not (_C["train"] & _C["val"]), "val still shares claims with train"
assert len(_C["train"]) == len(KEEP["train"]) and len(_C["val"]) == len(KEEP["val"]), "repeated claims remain in train/val"
print("✓ no claim is shared between train, val and test; no repeated claim in train or val")
print("\nSplit sizes after each step:")
display(pd.DataFrame(STEP_LOG).set_index("step"))
REMOVED_DF = pd.DataFrame(REMOVED, columns=["split", "row", "step", "reason", "label", "claim"])
if len(REMOVED_DF):
    print("Removed rows by split / step / gold label:")
    display(REMOVED_DF.groupby(["split", "step", "reason", "label"]).size().rename("rows").to_frame())
REMOVED_DF.to_csv(f"{OUT_DIR}/dedup_removed_items.csv", index=False, encoding="utf-8")
with open(f"{OUT_DIR}/dedup_report.json", "w", encoding="utf-8") as f:
    json.dump({"matching": "exact claim string", "steps": STEP_LOG, "kept_rows": KEEP}, f, indent=1)

S = {}
for s in SPLITS:
    S[s] = Data([RAW[s][i] for i in KEEP[s]])
    S[s].orig_row = list(KEEP[s])
    if s != "test":
        S[s].qid = list(KEEP[s])               # train/val embedding keys are the original positions

stats = {}
for s, d in S.items():
    nt = [len(e["Reasoning_traces"]) for e in d.items]
    nv = [len(V) for V in d.verdicts]
    stats[s] = {"rows": len(d.items), "unique claims": len(set(e["claim"] for e in d.items)),
                "repeated rows (kept)": len(d.items) - len(set(e["claim"] for e in d.items)),
                "traces min/mean/max": f"{min(nv)}/{np.mean(nv):.2f}/{max(nv)}",
                "rows #traces != #verdicts": sum(a != b for a, b in zip(nt, nv)),
                "unknown verdict values": sum(v not in CLASSES for V in d.verdicts for v in V),
                **{f"label {c}": int((d.y == i).sum()) for i, c in enumerate(CLASSES)},
                "unknown labels": int((d.y < 0).sum())}
display(pd.DataFrame(stats))
_tv_keys = {claim_key(e["claim"]) for s in ("train", "val") for e in S[s].items}
_mem = sum(claim_key(e["claim"]) in _tv_keys for e in S["test"].items)
print(f"test rows the ranker's exact-claim memory can still match after deduplication "
      f"(whitespace/case-normalised text): {_mem}/{len(S['test'].items)}")
for s, d in S.items():
    for i, e in enumerate(d.items):
        if len(e["Reasoning_traces"]) != len(e["Verdict_list"]):
            print(f"note: {s} row {d.orig_row[i]} has {len(e['Verdict_list'])} verdicts but {len(e['Reasoning_traces'])} trace "
                  f"texts - all verdicts are ranked; trace text is null where no text exists")

## 4. Embeddings
Layout of each npz: key `id_<original position>` (train/val) or `id_<query_id>` (test) → `(1 + #non-empty traces, dim)`,
row 0 = claim. Read one row at a time; only the claim vector, three verdict-group centroids and per-trace
cosines are kept, so RAM stays low.

In [ ]:
def npz_key(z, d, i):
    for k in (f"id_{d.qid[i]}",):
        if k in z.files:
            return k
    return None

def read_npz(path, d):
    z = np.load(path)
    claims, cents, coh, rep, ccos, has_traces, missing, mism = [], [], [], [], [], False, 0, 0
    diag = {"examples": [], "npz_rows": Counter(), "missing_rows": [],
            "n_keys": sum(f.startswith("id_") for f in z.files)}
    for i, e in enumerate(d.items):
        n_tr = len(d.verdicts[i])
        k = npz_key(z, d, i)
        if k is None:
            missing += 1
            diag["missing_rows"].append(i)
            claims.append(None)
            cents.append([None] * 3)
            coh.append([0.0] * 3)
            rep.append(np.zeros(n_tr))
            ccos.append(np.zeros(n_tr))
            continue
        a = z[k].astype(np.float32)
        a /= np.linalg.norm(a, axis=1, keepdims=True) + 1e-9
        claims.append(a[0])
        keep = [j for j, t in enumerate(e["Reasoning_traces"]) if str(t).strip()]
        mism += len(keep) != len(a) - 1
        diag["npz_rows"][len(a)] += 1
        if len(keep) != len(a) - 1 and len(diag["examples"]) < 5:
            diag["examples"].append(f"row {i}: key {k} has {len(a)} rows, the JSON has {len(keep)} non-empty traces "
                                    f"(expected {len(keep) + 1} rows)")
        keep = [j for j in keep[: len(a) - 1] if j < n_tr]
        tr = a[1 : 1 + len(keep)]
        has_traces |= len(keep) > 0
        r, c = np.full(n_tr, -1.0), np.full(n_tr, -1.0)
        if keep:
            c[keep] = tr @ a[0]
        V = [d.verdicts[i][j] for j in keep]
        cc, hh = [], []
        for cls in CLASSES:
            g = [j for j, v in enumerate(V) if v == cls]
            if g:
                m = tr[g].mean(0)
                hh.append(float(np.linalg.norm(m)))          # ~ mean pairwise cosine in the group
                m = m / (np.linalg.norm(m) + 1e-9)
                r[[keep[j] for j in g]] = tr[g] @ m          # representativeness (tie-break only)
                cc.append(m)
            else:
                hh.append(0.0)
                cc.append(None)
        cents.append(cc)
        coh.append(hh)
        rep.append(r)
        ccos.append(c)
    z.close()
    dim = next(c for c in claims if c is not None).shape[0]
    d.claim = np.vstack([c if c is not None else np.zeros(dim, np.float32) for c in claims])
    d.rep, d.claim_cos = rep, ccos
    d.cents, d.coh, d.has_traces = cents, np.array(coh), has_traces
    return missing, mism, dim, diag

def finish_embedding_features(splits, dim):
    """PCA fitted on TRAIN+VAL claims and group centroids (unsupervised), applied to every split."""
    fit = []
    for s in ("train", "val"):
        fit.append(splits[s].claim)
        fit += [c[None] for cc in splits[s].cents[:3000] for c in cc if c is not None]
    pca = PCA(dim, random_state=0).fit(np.vstack(fit))
    zero = np.zeros(dim)
    for d in splits.values():
        d.claim_pca = pca.transform(d.claim)
        if d.has_traces:
            rows = []
            for i, cc in enumerate(d.cents):
                sims = [float(c @ d.claim[i]) if c is not None else 0.0 for c in cc]
                pair = [float(cc[a] @ cc[b]) if cc[a] is not None and cc[b] is not None else 0.0
                        for a, b in ((0, 1), (0, 2), (1, 2))]
                blocks = [pca.transform(c[None])[0] if c is not None else zero for c in cc]
                rows.append(np.concatenate([sims, d.coh[i], pair, *blocks]))
            d.group = np.array(rows, np.float32)
        del d.cents, d.coh

USE_GROUP = False
if USE_EMB:
    for s in SPLITS:
        t = time.time()
        missing, mism, dim, diag = read_npz(NPZ_PATHS[s], S[s])
        print(f"{s:5s}: dim {dim} | missing keys {missing} | trace-row mismatches {mism} | {time.time() - t:.0f}s")
        if missing or mism:
            json_rows = Counter(1 + sum(bool(str(t).strip()) for t in e["Reasoning_traces"]) for e in S[s].items)
            print(f"\n  DIAGNOSIS for {s}")
            print(f"  npz : {NPZ_PATHS[s]}  ({os.path.getsize(NPZ_PATHS[s]):,d} bytes, {diag['n_keys']} id_ keys)")
            print(f"  json: {JSON_PATHS[s]}  ({len(S[s].items)} rows)")
            print(f"  rows per item in the npz     : {dict(sorted(diag['npz_rows'].items()))}")
            print(f"  1 + traces per item in JSON  : {dict(sorted(json_rows.items()))}")
            print(f"  JSON rows without an npz key : {diag['missing_rows'][:10]}")
            for x in diag["examples"]:
                print("  " + x)
            if set(diag["npz_rows"]) == {1}:
                print("  -> this npz holds ONLY claim vectors (1 row per item), not per-trace embeddings. "
                      "Point EMB_DIR at the per-trace files (row 0 = claim, rows 1.. = traces).")
            elif diag["n_keys"] != len(S[s].items):
                print(f"  -> the npz has {diag['n_keys']} items but the JSON has {len(S[s].items)}: the two were made "
                      f"from different copies of {os.path.basename(JSON_PATHS[s])}.")
        assert missing == 0, f"{s}: {missing} rows have no embedding - wrong npz for this JSON?"
        assert mism == 0, f"{s}: {mism} rows have a trace count different from the JSON - wrong npz for this JSON?"
    USE_GROUP = all(S[s].has_traces for s in SPLITS)
    finish_embedding_features(S, PCA_DIM)
print("feature blocks: counts, memory" + (", claim, knn" if USE_EMB else "") + (", group" if USE_GROUP else ""))

## 5. Label-probability model and ranking rules
Identical to `rank_traces.py`. Traces are ranked by verdict group in order of `p(label = c) / n_c`, then
within a group by closeness to the group centroid.

In [ ]:
def count_features(N):
    tot = N.sum(1, keepdims=True).clip(min=1)
    return np.hstack([N / tot, np.log1p(N), (N == 0).astype(float)])

def exact_memory(bank, q):
    """Label distribution of bank claims with identical text."""
    idx = {}
    for j in np.where(bank.y >= 0)[0]:
        idx.setdefault(bank.keys[j], []).append(j)
    hit, dist = np.zeros(len(q.y), bool), np.zeros((len(q.y), len(CLASSES)))
    for i, k in enumerate(q.keys):
        js = idx.get(k)
        if js:
            hit[i] = True
            dist[i] = np.bincount(bank.y[js], minlength=len(CLASSES)) / len(js)
    return hit, dist

def knn_features(bank, q, same, k=10, chunk=2048):
    """Top-1 / mean cosine to the k nearest training claims and their similarity-weighted labels."""
    ok = np.where(bank.y >= 0)[0]
    Cb, yb = bank.claim[ok], bank.y[ok]
    pos = np.full(len(bank.y), -1)
    pos[ok] = np.arange(len(ok))
    F = np.zeros((len(q.y), 2 + len(CLASSES)))
    for s in range(0, len(q.y), chunk):
        Sm = q.claim[s : s + chunk] @ Cb.T
        if same:  # leave-one-out: a claim may not vote for itself
            rows = np.arange(s, min(s + chunk, len(q.y)))
            cols = pos[rows]
            Sm[(rows - s)[cols >= 0], cols[cols >= 0]] = -1
        top = np.argpartition(-Sm, k, axis=1)[:, :k]
        for r, t in enumerate(top):
            sim = Sm[r, t]
            w = np.maximum(sim, 0) ** 4
            F[s + r, 0], F[s + r, 1] = sim.max(), sim.mean()
            F[s + r, 2:] = np.bincount(yb[t], weights=w, minlength=len(CLASSES)) / max(w.sum(), 1e-9)
    return F

def design(bank, q, cfg, same):
    X = [count_features(q.N)]
    if cfg["claim"]:
        X.append(q.claim_pca)
    if cfg["group"]:
        X.append(q.group)
    if cfg["knn"]:
        X.append(knn_features(bank, q, same))
    return np.hstack(X)

def fit_predict(tr, te, cfg, weights=None):
    """Fits configuration `cfg` on `tr`; returns label probabilities for `te`."""
    ok = tr.y >= 0
    Xtr, Xte = design(tr, tr, cfg, same=True), design(tr, te, cfg, same=False)
    sc = StandardScaler().fit(Xtr[ok])
    m = LogisticRegression(C=cfg["C"], max_iter=5000)
    m.fit(sc.transform(Xtr[ok]), tr.y[ok], sample_weight=None if weights is None else weights[ok])
    P = np.zeros((len(te.y), len(CLASSES)))
    P[:, m.classes_] = m.predict_proba(sc.transform(Xte))
    hit, dist = exact_memory(tr, te)
    P[hit] = 0.9 * dist[hit] + 0.1 * P[hit]
    return P

def shift_weights(tv, te, clip):
    """Importance weights p(test|x)/p(train|x) from a domain classifier on counts (+ claim PCA)."""
    blocks = lambda d: np.hstack([count_features(d.N)] + ([d.claim_pca] if d.claim_pca is not None else []))
    X = np.vstack([blocks(tv), blocks(te)])
    dom = np.r_[np.zeros(len(tv.y)), np.ones(len(te.y))]
    sc = StandardScaler().fit(X)
    m = LogisticRegression(C=0.1, max_iter=5000).fit(sc.transform(X), dom)
    p = m.predict_proba(sc.transform(blocks(tv)))[:, 1]
    w = np.clip(p / (1 - p) * len(tv.y) / len(te.y), 0, clip)
    return w / w.mean()

def rank_claim(p, verdicts, rep=None):
    """Trace order: verdict groups by p_c / n_c, within a group by representativeness."""
    n = Counter(verdicts)
    prio = {v: (p[CLASSES.index(v)] / n[v] if v in CLASSES else -1.0) for v in n}
    rep = np.zeros(len(verdicts)) if rep is None else rep
    return sorted(range(len(verdicts)), key=lambda i: (-prio[verdicts[i]], -rep[i], i))

def orders_from_probs(P, d):
    reps = d.rep or [None] * len(d.y)
    return [rank_claim(p, V, r) for p, V, r in zip(P, d.verdicts, reps)]

def baseline_orders(d, kind, seed=SEED):
    rng = np.random.default_rng(seed)
    out = []
    for i, V in enumerate(d.verdicts):
        n = Counter(V)
        if kind == "random":
            out.append(list(rng.permutation(len(V))))
        elif kind == "majority_first":
            out.append(sorted(range(len(V)), key=lambda j: (-n[V[j]], j)))
        elif kind == "minority_first":
            out.append(sorted(range(len(V)), key=lambda j: (n[V[j]], j)))
        elif kind == "oracle":
            out.append(sorted(range(len(V)), key=lambda j: (V[j] != d.gold[i], j)))
        elif kind == "claim_cosine":
            out.append(list(np.argsort(-d.claim_cos[i], kind="stable")))
    return [[int(j) for j in o] for o in out]

def recall_at_k(order, verdicts, gold, k):
    total = sum(v == gold for v in verdicts)
    return sum(verdicts[i] == gold for i in order[:k]) / total if total else 0.0

def evaluate(orders, d):
    """Mean Recall@1..K over ALL rows (no deduplication)."""
    return {k: float(np.mean([recall_at_k(o, V, g, k) for o, V, g in zip(orders, d.verdicts, d.gold)]))
            for k in range(1, K + 1)}

## 6. Feature-block selection by 5-fold CV on train+val
Every candidate's out-of-fold probabilities are kept. The selected candidate's out-of-fold probabilities
become the train/val rankings of `pn_selected`, and the counts-only candidate's become those of `pn_counts`.

In [ ]:
TV = concat(S["train"], S["val"])
TE = S["test"]
N_TR = len(S["train"].y)

cands = [dict(claim=False, group=False, knn=False, C=1.0)]
if USE_EMB:
    for claim in (False, True):
        for group in ((False, True) if USE_GROUP else (False,)):
            for knn in (False, True):
                if claim or group or knn:
                    cands += [dict(claim=claim, group=group, knn=knn, C=C) for C in (0.01, 0.1)]
FOLDS = list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED).split(np.zeros(len(TV.y)), np.maximum(TV.y, 0)))

def oof_probs(cfg, weights=None):
    P = np.zeros((len(TV.y), len(CLASSES)))
    for a, b in FOLDS:
        P[b] = fit_predict(TV.take(a), TV.take(b), cfg, None if weights is None else weights[a])
    return P

OOF, CV_SCORES = [], []
for cfg in cands:
    t = time.time()
    OOF.append(oof_probs(cfg))
    CV_SCORES.append(evaluate(orders_from_probs(OOF[-1], TV), TV)[K])
    print(f"  CV R@{K} {CV_SCORES[-1]:.4f}  {cfg}  ({time.time() - t:.0f}s)", flush=True)
top = max(CV_SCORES)
size = lambda c: c["claim"] + c["group"] + c["knn"]
BEST_I = min((i for i, s in enumerate(CV_SCORES) if s >= top - MIN_GAIN), key=lambda i: size(cands[i]))
BEST = cands[BEST_I]
print(f"selected {BEST} (CV {CV_SCORES[BEST_I]:.4f}, best {top:.4f})")

## 7. Rank every row with every method
*Test:* p/n models refit on all of train+val. *Train/val:* out-of-fold probabilities from §6. The shift
weights (train+val vs test) are computed once and also used inside the folds.

In [ ]:
PROBS_TEST = {"pn_counts": fit_predict(TV, TE, cands[0]),
              "pn_selected": fit_predict(TV, TE, BEST)}
PROBS_TV   = {"pn_counts": OOF[0], "pn_selected": OOF[BEST_I]}
if SHIFT_CLIP > 0:
    W = shift_weights(TV, TE, SHIFT_CLIP)
    print(f"shift weights: min {W.min():.3f} | max {W.max():.3f} | effective n {W.sum() ** 2 / (W ** 2).sum():.0f}/{len(W)}")
    PROBS_TEST["pn_selected_shift"] = fit_predict(TV, TE, BEST, weights=W)
    PROBS_TV["pn_selected_shift"]   = oof_probs(BEST, W)

METHODS = {
    "random":            "Random order",
    "majority_first":    "Majority verdict first",
    "minority_first":    "Minority verdict first",
    "claim_cosine":      "Claim-trace cosine",
    "pn_counts":         "p/n, counts (+ exact claim memory)",
    "pn_selected":       "p/n + CV-selected blocks (ours)",
    "pn_selected_shift": "p/n + covariate-shift weighting (‡)",
    "oracle":            "Oracle (gold label known)",
}
if not USE_EMB:
    METHODS.pop("claim_cosine")
if SHIFT_CLIP <= 0:
    METHODS.pop("pn_selected_shift")

def split_probs(m, s):
    if m not in PROBS_TEST:
        return None
    if s == "test":
        return PROBS_TEST[m]
    return PROBS_TV[m][:N_TR] if s == "train" else PROBS_TV[m][N_TR:]

ORDERS, PROBS = {}, {}
for m in METHODS:
    for s in SPLITS:
        P = split_probs(m, s)
        PROBS[m, s] = P
        ORDERS[m, s] = baseline_orders(S[s], m) if P is None else [[int(j) for j in o] for o in orders_from_probs(P, S[s])]

# every order must be a permutation of all traces of its row
for (m, s), orders in ORDERS.items():
    assert len(orders) == len(S[s].items), (m, s)
    assert all(sorted(o) == list(range(len(V))) for o, V in zip(orders, S[s].verdicts)), (m, s)

rows = []
for m, name in METHODS.items():
    for s in SPLITS:
        r = evaluate(ORDERS[m, s], S[s])
        rows.append({"method": m, "name": name, "split": s, "rows": len(S[s].items), **{f"R@{k}": round(r[k], 4) for k in r}})
RESULTS = pd.DataFrame(rows)
RESULTS.to_csv(f"{OUT_DIR}/recall_at_k.csv", index=False)
print(f"Recall@{K} over all rows of each split (train/val deduplicated in §3, test complete); train/val = out-of-fold")
display(RESULTS.pivot(index="method", columns="split", values=f"R@{K}").loc[list(METHODS), list(SPLITS)])
print("Test, Recall@1..5:")
display(RESULTS[RESULTS.split == "test"].set_index("method").drop(columns=["split", "rows"]))

## 8. Save the ordered traces
One file per method and split: `rankings/<method>/<split>.json`. Each file is a list with one record per
row, in the original file order:

```json
{
  "split": "test", "row": 0, "query_id": 0,          // row = position in the released file
  "claim": "...", "label": "False",
  "label_probs": {"True": 0.1, "False": 0.8, "Conflicting": 0.1},   // p/n methods only, else null
  "recall@5": 1.0,
  "order": [3, 0, 7, ...],          // trace indices (into Reasoning_traces / Verdict_list), best first
  "score_list": [19.0, 2.0, ...],   // scorer.py format: score per trace in original order, higher = ranked higher
  "ranked_traces": [
    {"rank": 1, "trace_index": 3, "verdict": "False", "relevant": true, "text": "..."},
    ...
  ]
}
```
`relevant` = the trace's verdict equals the gold label (the scorer's definition).

In [ ]:
def build_records(d, split, orders, P):
    recs = []
    for i, (e, o) in enumerate(zip(d.items, orders)):
        traces, V, n = e["Reasoning_traces"], e["Verdict_list"], len(o)
        score, ranked = [0.0] * n, []
        for r, j in enumerate(o):
            score[j] = float(n - r)
            t = {"rank": r + 1, "trace_index": j, "verdict": V[j], "relevant": d.verdicts[i][j] == d.gold[i]}
            if SAVE_TRACE_TEXT:
                t["text"] = traces[j] if j < len(traces) else None
            ranked.append(t)
        recs.append({
            "split": split, "row": d.orig_row[i], "query_id": e.get("query_id"),
            "claim": e["claim"], "label": e.get("label", e.get("Label")),
            "label_probs": None if P is None else {c.capitalize(): round(float(p), 5) for c, p in zip(CLASSES, P[i])},
            f"recall@{K}": recall_at_k(o, d.verdicts[i], d.gold[i], K),
            "order": o, "score_list": score, "ranked_traces": ranked,
        })
    return recs

SAVED = []
for m in METHODS:
    os.makedirs(f"{OUT_DIR}/rankings/{m}", exist_ok=True)
    for s in SPLITS:
        path = f"{OUT_DIR}/rankings/{m}/{s}.json"
        with open(path, "w", encoding="utf-8") as f:
            json.dump(build_records(S[s], s, ORDERS[m, s], PROBS[m, s]), f, ensure_ascii=False)
        SAVED.append(path)
        print(f"  {path}  ({os.path.getsize(path) / 1e6:.1f} MB)")

meta = {
    "data": JSON_PATHS, "embeddings": NPZ_PATHS, "deduplication": {"matching": "exact claim string", "steps": STEP_LOG,
                      "removed": {s: int((REMOVED_DF.split == s).sum()) for s in SPLITS}},
    "rows": {s: len(S[s].items) for s in SPLITS},
    "train_val_rankings": f"out-of-fold, {N_FOLDS}-fold StratifiedKFold(random_state={SEED}) over train+val",
    "feature_blocks_available": {"embeddings": USE_EMB, "group": USE_GROUP},
    "cv_candidates": [{"cfg": c, f"cv_R@{K}": round(sc, 4)} for c, sc in zip(cands, CV_SCORES)],
    "selected": BEST, "pca_dim": PCA_DIM, "min_gain": MIN_GAIN, "shift_clip": SHIFT_CLIP,
    "methods": METHODS, "files": SAVED,
    "recall_at_k_all_rows": RESULTS.to_dict(orient="records"),
}
with open(f"{OUT_DIR}/run_meta.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)
print(f"saved {len(SAVED)} ranking files + recall_at_k.csv + run_meta.json in {OUT_DIR}")

## 9. Check the saved files
Each file is reloaded, and Recall@5 is recomputed from it in two ways: from `score_list`, the way the
official scorer reads a submission but **without** its deduplication, and from `ranked_traces`. Both must
match the table in §7.

In [ ]:
check = []
for m in METHODS:
    for s in SPLITS:
        with open(f"{OUT_DIR}/rankings/{m}/{s}.json", encoding="utf-8") as f:
            data = json.load(f)
        assert len(data) == len(S[s].items), f"{m}/{s}: row count changed"
        r_score, r_ranked = [], []
        for e in data:
            label = e["label"].lower()
            verdicts = [norm(t["verdict"]) for t in sorted(e["ranked_traces"], key=lambda t: t["trace_index"])]
            ranked = sorted(range(len(e["score_list"])), key=lambda i: e["score_list"][i], reverse=True)
            total = sum(v == label for v in verdicts)
            r_score.append(sum(verdicts[i] == label for i in ranked[:K]) / total if total else 0.0)
            r_ranked.append(sum(t["relevant"] for t in e["ranked_traces"][:K]) / total if total else 0.0)
        expected = RESULTS[(RESULTS.method == m) & (RESULTS.split == s)][f"R@{K}"].item()
        check.append({"method": m, "split": s, "rows": len(data), "from score_list": round(np.mean(r_score), 4),
                      "from ranked_traces": round(np.mean(r_ranked), 4), "table": expected})
        assert abs(np.mean(r_score) - expected) < 1e-4 and abs(np.mean(r_ranked) - expected) < 1e-4, check[-1]
display(pd.DataFrame(check))
print(f"all files verified | total time {(time.time() - T0) / 60:.1f} min")

## 10. Example: top 5 traces of one test row

In [ ]:
with open(f"{OUT_DIR}/rankings/pn_selected/test.json", encoding="utf-8") as f:
    ex = json.load(f)[0]
print(f"claim : {ex['claim']}\nlabel : {ex['label']} | probs {ex['label_probs']} | recall@{K} {ex[f'recall@{K}']:.3f}")
for t in ex["ranked_traces"][:K]:
    text = (t.get("text") or "").replace("\n", " ")
    print(f"\n#{t['rank']}  trace {t['trace_index']}  verdict={t['verdict']}  relevant={t['relevant']}\n   {text[:300]}...")